#Section 1 - Environment Setup

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_dir = "/content/drive/MyDrive/RemoteSensing_Internship"
os.makedirs(project_dir, exist_ok=True)
print("Project directory:", project_dir)

In [ ]:
%cd /content/drive/MyDrive/RemoteSensing_Internship

In [ ]:
!pip install -q timm kagglehub

#Section 2 - Dataset Download and Structure Check (NWPU-RESISC45)

In [ ]:
import kagglehub

path = kagglehub.dataset_download("aqibrehmanpirzada/nwpuresisc45")
print("Dataset downloaded to:", path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    print("  Folders:", dirs[:10])
    print("  Files:", files[:5])
    print("-" * 60)

In [ ]:
train_dir = os.path.join(path, "Dataset", "train", "train")
test_dir = os.path.join(path, "Dataset", "test", "test")

print("Train dir:", train_dir)
print("Test dir:", test_dir)

In [ ]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import ConcatDataset, DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_part = ImageFolder(root=train_dir, transform=transform)
test_part = ImageFolder(root=test_dir, transform=transform)

# Sanity check: both folders must map to the exact same 45 classes in the same order
assert train_part.classes == test_part.classes, "Class mismatch between train and test folders!"

dataset = ConcatDataset([train_part, test_part])
class_names = train_part.classes   # keep this handy — ConcatDataset has no .classes attribute

print("Total Images:", len(dataset))
print("Number of Classes:", len(class_names))
for i, class_name in enumerate(class_names):
    print(f"{i}: {class_name}")

In [ ]:
image, label = dataset[0]
print("Image Shape:", image.shape)
print("Label Index:", label)
print("Class:", class_names[label])

#Section 3 - Train/Val/Test Split and DataLoaders


In [ ]:
dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))
print("Testing Images:", len(test_dataset))

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print("Image Batch Shape:", images.shape)
print("Label Batch Shape:", labels.shape)

In [ ]:
import matplotlib.pyplot as plt

img = images[0].permute(1, 2, 0)
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])
img = img * std + mean
img = img.clamp(0, 1)

plt.imshow(img)
plt.title(f"Class: {class_names[labels[0]]}")
plt.axis('off')
plt.show()

#Section 4 - MambaOut Model Setup

In [ ]:
import timm

NUM_CLASSES = len(class_names)   # 45 for NWPU-RESISC45

model = timm.create_model(
    "mambaout_tiny",
    pretrained=False,
    num_classes=NUM_CLASSES
)

print(model)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(device)

#Section 5 - Training Setup

In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
print(criterion)

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print(optimizer)

In [ ]:
# 45 classes is a bigger task than EuroSAT's 10 — step the LR down a bit later
# so the model has more time to converge before the drop
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)

print(scheduler)

#Section 6 - Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
EPOCHS = 15   # more than EuroSAT's 5 — 45-way classification needs more time to converge

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

In [ ]:
best_accuracy = 0
import time

start_time = time.time()

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    scheduler.step()

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc:.2f}%")
    print("-"*40)

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), "best_mambaout_nwpu45.pth")

end_time = time.time()
training_time = end_time - start_time

print(f"Training Time: {training_time:.2f} seconds")
print("Training Finished!")
print("Best Validation Accuracy:", best_accuracy)

Epoch 1/15
Train Loss : 2.6415
Train Accuracy : 26.56%
Validation Loss : 1.9079
Validation Accuracy : 44.92%
----------------------------------------
Epoch 2/15
Train Loss : 1.6235
Train Accuracy : 51.87%
Validation Loss : 1.3734
Validation Accuracy : 58.98%
----------------------------------------
Epoch 3/15
Train Loss : 1.1156
Train Accuracy : 65.78%
Validation Loss : 1.1377
Validation Accuracy : 65.30%
----------------------------------------
Epoch 4/15
Train Loss : 0.8046
Train Accuracy : 74.61%
Validation Loss : 0.9223
Validation Accuracy : 72.32%
----------------------------------------
Epoch 5/15
Train Loss : 0.5765
Train Accuracy : 81.62%
Validation Loss : 0.8557
Validation Accuracy : 74.13%
----------------------------------------
Epoch 6/15
Train Loss : 0.3908
Train Accuracy : 87.22%
Validation Loss : 0.7877
Validation Accuracy : 77.08%
----------------------------------------
Epoch 7/15
Train Loss : 0.2473
Train Accuracy : 91.98%
Validation Loss : 0.7676
Validation Accuracy 

#Section 7 - Model Evaluation

In [ ]:
model.load_state_dict(torch.load("best_mambaout_nwpu45.pth"))
model.eval()
print("Best model loaded successfully!")

In [ ]:
test_loss, test_accuracy = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
import torch.nn.functional as F

all_preds = []
all_labels = []
all_probs = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm,
    annot=False,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("MambaOut — NWPU-RESISC45 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.()
plt.show()tight_layout

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')
auc = roc_auc_score(all_labels, np.array(all_probs), multi_class='ovr', average='macro')

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"AUC       : {auc:.4f}")

#Section 8 - Training Curves


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, marker='o', label="Train Loss")
plt.plot(val_losses, marker='o', label="Validation Loss")
plt.title("MambaOut NWPU-45 — Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_accuracies, marker='o', label="Train Accuracy")
plt.plot(val_accuracies, marker='o', label="Validation Accuracy")
plt.title("MambaOut NWPU-45 — Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.show()

#Section 9 — Results Summary


In [ ]:
print("="*50)
print("MambaOut Results — NWPU-RESISC45")
print("="*50)
print(f"Training Time : {training_time:.2f} sec")
print(f"Training Loss : {train_losses[-1]:.4f}")
print(f"Validation Loss : {val_losses[-1]:.4f}")
print(f"Training Accuracy : {train_accuracies[-1]:.2f}%")
print(f"Validation Accuracy : {val_accuracies[-1]:.2f}%")
print(f"Test Accuracy : {test_accuracy:.2f}%")

print(f"Precision : {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC : {auc:.4f}")